In [1]:
# general
import numpy as np
import pandas as pd
import geopandas as gpd
import sys
import gc
import glob
import os
import swifter
from h3 import h3
import requests
import zipfile
import io
import json
from shapely import Point, Polygon, buffer

# plotting and output
from tqdm.auto import tqdm
# import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings(action='ignore')
tqdm.pandas()

In [2]:
label_columns = ['cropland', 'landcover', 'cropgroup', 'croptype', 'sampling_croptype']
wc2ec_map = pd.read_csv('resources/wc2eurocrops_map.csv')
ec_map = pd.read_csv('resources/eurocrops_map_sampling_edition.csv', index_col='ec_code')

In [3]:
local_path = '/home/cbutsko/Desktop/cbutsko_experiments/tmp/'
out_dir = '/home/cbutsko/Desktop/cbutsko_experiments/rdm_sampling_files/'

tdir = '/home/cbutsko/Desktop/cbutsko_experiments/rdm_croptype_stats/'
files_lst = glob.glob('{}*items'.format(tdir))
files_lst = [xx for xx in files_lst if int(xx.split('/')[-1][:4])>2017]

shape_fnames = [xx for xx in glob.glob('{}*'.format(local_path)) if xx.endswith('.shp')]
gpkg_fnames = [xx for xx in glob.glob('{}*'.format(local_path)) if xx.endswith('.gpkg')]

In [ ]:
exception_files = []
max_samples_per_cell = 50
cols_to_export = ['sampleID','centroid','sampling_croptype_name','h3_cell_res_3','ec_code']

pbar = tqdm(total=len(files_lst))
for tfname in files_lst:
    pbar.update(1)

    with open(tfname) as f:
        tdata = json.load(f)

    try:
        turl = [xx['value'] for xx in tdata if xx['name']=='CollectionDownloadUrl'][0]
    except:
        exception_files.append(tfname)
        continue
    
    ref_id = turl.split('/')[-1].split('.')[0]
    if len([xx for xx in shape_fnames if ref_id in xx])==0: 
        r = requests.get(turl)
        tzip = zipfile.ZipFile(io.BytesIO(r.content))
        try:
            tzip.extractall(path=local_path)
        except:
            exception_files.append(tfname)
            break
        shape_fname = ['{}{}'.format(local_path, xx) for xx in tzip.namelist() if '.shp' in xx][0]
        ref_id = '_'.join(shape_fname.split('/')[-1].split('_')[:3])
    else:
        shape_fname = [xx for xx in shape_fnames if ref_id in xx][0]

    out_fname = '{}{}_grid_samples.gpkg'.format(out_dir, ref_id)
    if os.path.isfile(out_fname):
        continue

    original_data = gpd.read_file(shape_fname)
    original_data['centroid'] = original_data['geometry'].centroid
    original_data['lat'] = original_data['centroid'].swifter.apply(lambda xx: xx.y, axis=1)
    original_data['lon'] = original_data['centroid'].swifter.apply(lambda xx: xx.x, axis=1)
    original_data['h3_cell_res_3'] = original_data.swifter.apply(lambda xx: h3.geo_to_h3(
        xx['lat'],
        xx['lon'],
        resolution=3
        ), axis=1)
    h3_cells_lst = original_data['h3_cell_res_3'].unique()
    
    original_data['CT'] = original_data['CT'].astype(int)
    original_data['LC'] = original_data['LC'].astype(int) 
    original_data['CT'].replace(0, np.nan, inplace=True)
    original_data['CT'].fillna(original_data['LC'], inplace=True)
    original_data['ec_code'] = original_data['CT'].map(wc2ec_map.set_index('croptype')['ec_code'])
    for tlevel in label_columns:
        original_data[tlevel] = original_data['ec_code'].map(ec_map['{}_label'.format(tlevel)]).astype('int32')
        original_data['{}_name'.format(tlevel)] = original_data['ec_code'].map(ec_map['{}_name'.format(tlevel)])

    grid_samples = []
    crops_to_sample = list(original_data['sampling_croptype_name'].unique())
    crops_to_sample = [xx for xx in crops_to_sample if xx not in ['cropland_unspecified','unknown']]
    original_data['is_grid_sample'] = False
    
    for h3_cell in h3_cells_lst:
        for crop in crops_to_sample:
            h3_all_samples = original_data[
                (original_data['h3_cell_res_3']==h3_cell) & 
                (original_data['sampling_croptype_name']==crop)
                ]
            if len(h3_all_samples)==0:
                continue
            if len(h3_all_samples)<=max_samples_per_cell:
                grid_samples.extend(list(h3_all_samples['sampleID'].values))
                continue
            h3_cell_buffered = buffer(Polygon(h3.h3_to_geo_boundary(h3_cell)), distance=-0.1)
            coords_grid = np.mgrid[
                        (np.min(h3_cell_buffered.envelope.exterior.coords.xy[1])+0.01):(np.max(h3_cell_buffered.envelope.exterior.coords.xy[1])-0.01):complex(0,np.ceil(np.sqrt(max_samples_per_cell))), 
                        (np.min(h3_cell_buffered.envelope.exterior.coords.xy[0])+0.01):(np.max(h3_cell_buffered.envelope.exterior.coords.xy[0])-0.01):complex(0,np.ceil(np.sqrt(max_samples_per_cell)))].reshape(2,-1).T
            coords_grid = coords_grid[[Polygon(h3.h3_to_geo_boundary(h3_cell)).contains(Point(xx[::-1])) for xx in coords_grid]]
            _grid_samples = []
            for grid_point in coords_grid:
                tsample = ((h3_all_samples['lon'] - grid_point[0]).abs() + (h3_all_samples['lat'] - grid_point[1]).abs()).argmin()
                _grid_samples.append(h3_all_samples.iloc[tsample]['sampleID'])
            grid_samples.extend(_grid_samples)

    original_data['is_grid_sample'] = original_data['sampleID'].isin(grid_samples)

    tdata = original_data[(original_data['is_grid_sample'])][cols_to_export]

    if 'valtime' in original_data.columns:
        tdata['valid_date'] = original_data['valtime']
    if 'AcquiDate' in original_data.columns:
        tdata['valid_date'] = original_data['AcquiDate']
    if 'today' in original_data.columns:
        tdata['valid_date'] = original_data['today']

    tdata['ref_id'] = shape_fname.split('/')[-1].split('.')[0]
    tdata['extract'] = True
    tdata['confidence'] = np.nan
    tdata['croptype_label'] = tdata['ec_code'].str.replace('-','').astype(int)
    tdata['valid_date'] = pd.to_datetime(tdata['valid_date'])

    tdata = gpd.GeoDataFrame(tdata, geometry='centroid')
    tdata.to_file(out_fname, driver='GPKG')

    del original_data
    del tdata
    gc.collect()

    files2clean = [xx for xx in glob.glob('{}*'.format(local_path)) if ref_id in xx]
    for tfile in files2clean:
        os.remove(tfile)
        gc.collect()